In [1]:
#!/usr/bin/env Rscript

#############################
# Run_cluster_tree_num_fixed.R
# Description: 根据 resolution 计算聚类数，并生成 cluster number 图和 clustree 图
# 直接使用固定 RDS 文件和输出路径
# 完全不依赖 ggalt，使用原生 ggplot2 绘图
#############################
library(Seurat)
library(ggplot2)
library(cowplot)
library(Matrix)
library(dplyr)
library(patchwork)
library(clustree, lib.loc = "/data/work/renv/library/R-4.2/x86_64-conda-linux-gnu")
library(data.table, lib.loc = "/opt/conda/lib/R/library")

# RDS 文件和 SampleID
rds_file <- "/data/work/RNA_AraSeed/rds/cot/cot_after_QC_doublet_filtered.rds"  #🥰
SampleID <- "cot"                                                         #🥰

# 输出路径 
output_dir <- "/data/work/RNA_AraSeed/QC/cot"             #🥰

# 如果输出路径不存在，自动创建
if (!dir.exists(output_dir)) {
  dir.create(output_dir, recursive = TRUE)
}

# 设置 resolution 范围
resolutions <- seq(0.1, 2, 0.1)
cluster_num_list <- c()

# 读取 RDS 数据
cat("Reading RDS file:", rds_file, "\n")
data <- readRDS(rds_file)

# 输出细胞总数检查
cat("Number of cells in RDS:", ncol(data), "\n")

# 循环计算不同 resolution 下的聚类数
for (res in resolutions) {
  cat("Processing resolution:", res, "\n")
  data <- FindClusters(data, resolution = res, verbose = FALSE)
  cluster_num_list <- c(cluster_num_list, length(unique(data$seurat_clusters)))
}

# 保存聚类数数据
cluster_df <- data.frame(resolution = resolutions, cluster_num = cluster_num_list)

# 打印结果
cat("\nResolution vs Cluster number:\n")
print(cluster_df)

# 绘制 cluster number 图（使用原生 ggplot2，不依赖 ggalt）
cat("\nGenerating cluster number plot...\n")

cluster_plot <- ggplot(cluster_df, aes(x = resolution, y = cluster_num)) +
  # 柱状图
  geom_bar(stat = "identity", fill = "steelblue", alpha = 0.7, width = 0.08) +
  # 红色实线连接实际数据点（修改1: size -> linewidth）
  geom_line(color = "darkred", linewidth = 1.2) +
  # 红色圆点标记实际数据点
  geom_point(color = "darkred", size = 2.5, shape = 16) +
  # 平滑趋势线（替代 geom_xspline）（修改2: size -> linewidth）
  stat_smooth(method = "loess", se = FALSE, color = "orange", 
              linetype = "dashed", linewidth = 0.9, span = 0.6) +
  # 坐标轴标签
  labs(x = "Resolution", y = "Number of clusters") +
  # 主题设置
  theme_bw(base_size = 12) +
  theme(
    axis.title = element_text(face = "bold", size = 12),
    axis.text = element_text(size = 10),
    plot.title = element_text(hjust = 0.5, face = "bold"),
    panel.grid.minor = element_blank()
  )

# 保存 cluster number 图
pdf(file.path(output_dir, paste0(SampleID, ".cluster_num.pdf")), width = 8, height = 6)
print(cluster_plot)
dev.off()
cat("Cluster number plot saved to:", file.path(output_dir, paste0(SampleID, ".cluster_num.pdf")), "\n")

# 绘制 clustree 图（修改3: 移除重复的 scale_edge_color_continuous）
cat("\nGenerating clustree plot...\n")
clus_tree <- clustree(data,
                      edge_colour = "grey80",  # 直接在函数中设置边缘颜色
                      node_colour = "steelblue") +  # 设置节点颜色
  theme(legend.position = "right")

pdf(file.path(output_dir, paste0(SampleID, ".cluster_tree.pdf")), width = 12, height = 20)
print(clus_tree)
dev.off()

cat("\nDone! All plots saved in:", output_dir, "\n")
cat("Files created:\n")
cat("  -", paste0(SampleID, ".cluster_num.pdf"), "\n")
cat("  -", paste0(SampleID, ".cluster_tree.pdf"), "\n")
# 绘制 Elbow plot（保存到输出目录）
pdf(file.path(output_dir, paste0(SampleID, ".elbow_plot.pdf")))
ElbowPlot(data, ndims = 30)
dev.off()
cat("Elbow plot saved to:", file.path(output_dir, paste0(SampleID, ".elbow_plot.pdf")), "\n")

# 或者用更精确的方法：计算累计方差贡献率
pct <- data[["pca"]]@stdev / sum(data[["pca"]]@stdev) * 100
cum_pct <- cumsum(pct)
dims_selected <- which(cum_pct >= 90)[1]  # 取累计贡献率达到90%的维度
cat("选择的维度数:", dims_selected, "\n")

cat("Number of cells in RDS:", ncol(data), "\n")

Attaching SeuratObject


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘patchwork’


The following object is masked from ‘package:cowplot’:

    align_plots


Loading required package: ggraph


Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last




Reading RDS file: /data/work/RNA_AraSeed/rds/cot/cot_after_QC_doublet_filtered.rds 
Number of cells in RDS: 35332 
Processing resolution: 0.1 
Processing resolution: 0.2 
Processing resolution: 0.3 
Processing resolution: 0.4 
Processing resolution: 0.5 
Processing resolution: 0.6 
Processing resolution: 0.7 
Processing resolution: 0.8 
Processing resolution: 0.9 
Processing resolution: 1 
Processing resolution: 1.1 
Processing resolution: 1.2 
Processing resolution: 1.3 
Processing resolution: 1.4 
Processing resolution: 1.5 
Processing resolution: 1.6 
Processing resolution: 1.7 
Processing resolution: 1.8 
Processing resolution: 1.9 
Processing resolution: 2 

Resolution vs Cluster number:
   resolution cluster_num
1         0.1           8
2         0.2          10
3         0.3          13
4         0.4          15
5         0.5          17
6         0.6          18
7         0.7          19
8         0.8          20
9         0.9          22
10        1.0          22
11        1.

`geom_smooth()` using formula = 'y ~ x'


pdf 
  2

Cluster number plot saved to: /data/work/RNA_AraSeed/QC/cot/cot.cluster_num.pdf 

Generating clustree plot...


pdf 
  2


Done! All plots saved in: /data/work/RNA_AraSeed/QC/cot 
Files created:
  - cot.cluster_num.pdf 
  - cot.cluster_tree.pdf 


pdf 
  2

Elbow plot saved to: /data/work/RNA_AraSeed/QC/cot/cot.elbow_plot.pdf 
选择的维度数: 26 
Number of cells in RDS: 35332 
